# EnderLeaf script preparation

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

## Imports

In [ ]:
from typing import Literal
import time
import io

from rich.pretty import pprint

import numpy as np

from picamera2.encoders import JpegEncoder
from picamera2.outputs import FileOutput
from picamera2 import Picamera2, Preview

from enderscope.scan_patterns import snake, plot_path
from enderscope.serial import list_ports, Stage
from enderscope.bed import bed
from enderscope.enderlights import Enderlights

import panel as pn

from enderleaf.image import Rectangle, lap_var, to_pil
from enderleaf.tools import time_method
from enderleaf.preview_panel import preview, get_last_still, StillFolders
from enderscope.enderlights_pi import Enderlights, cycle

In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

In [ ]:
%matplotlib widget

## Setup

## Constants

In [ ]:
TEMPLATE_LENGTH = 210
TEMPLATE_SIZE = (TEMPLATE_LENGTH, TEMPLATE_LENGTH)
ROW_COUNT, COL_COUNT = 9, 9
LEAF_DIAM = 17
CAM_RES = (4608, 2592)
CROP_TOP = 600
CROP_BOTTOM = 550
CROP_LEFT = 1300
CROP_RIGHT = 1470

## Scan Pattern

In [ ]:
positions = snake(cols=COL_COUNT, rows=ROW_COUNT) * [
    # steps
    TEMPLATE_LENGTH / COL_COUNT,
    TEMPLATE_LENGTH / ROW_COUNT,
] + [
    # origin
    TEMPLATE_LENGTH / COL_COUNT / 2,
    TEMPLATE_LENGTH / ROW_COUNT / 2,
]
plot_path(
    positions,
    title="snake scan",
    field=(LEAF_DIAM + 3, LEAF_DIAM + 3),
    selected_rectangles=[17, 41, 55, 81],
    circle_diam=LEAF_DIAM,
)

In [ ]:
positions[0]

## 3D Virtual Scan

In [ ]:
s = Stage("virtual", 115200)

In [ ]:
s.home()

In [ ]:
# s.move_position((bed.x_max / 2, bed.y_max / 2, bed.global_height))
for p in positions:
    s.move_position(np.append(p, bed.individual_height))

## 3D Scan

In [ ]:
def acquire_image(
    stage: Stage,
    lights: Enderlights,
    pos,
    read_qr: bool,
    crop_data: Rectangle | None = None,
):
    stage.move_position(pos)
    stage.finish_moves()
    lights.shutter(True)
    image = preview().do_capture_array(crop_data=crop_data)
    lights.shutter(False)
    return image

In [ ]:
def get_best_z(
    stage,
    lights,
    pos,
    crop_data: Rectangle,
    min_rel_z: float = 5.0,
    max_rel_z: float = 5.0,
):
    zrange = np.array(range(min_rel_z, max_rel_z, 1))
    mxScore = -1
    bestZ = 0
    lights.shutter(True)

    for z in zrange:
        stage.move_position([pos[0], pos[1], z + pos[2]])
        stage.finish_moves()
        img = preview().do_capture_array(crop_data=crop_data)
        grayImage = (
            np.float32(img[:, :, 0])
            + np.float32(img[:, :, 1])
            + np.float32(img[:, :, 2])
        ) / 3
        score = lap_var(grayImage)
        if score > mxScore:
            mxScore = score
            bestZ = z
    lights.shutter(False)

    stage.move_position([pos[0], pos[1], pos[2]])
    stage.finish_moves()

    return bestZ + pos[2]

In [ ]:
@time_method
def run_job(stage, lights, speed, start_height):
    # s.set_speed(speed=speed)
    # acquire_image(
    #     stage=stage,
    #     lights=lights,
    #     camera=camera,
    #     pos=(bed.x_max / 2, bed.y_max / 2, bed.global_height),
    #     read_qr=True,
    #     resolution=CAM_RES,
    # )

    z = get_best_z(
        stage=stage,
        lights=lights,
        pos=[*positions[0], start_height],
        crop_data=Rectangle(
            top=CROP_TOP,
            bottom=CAM_RES[1] - CROP_BOTTOM,
            left=CROP_LEFT,
            right=CAM_RES[0] - CROP_RIGHT,
        ),
        min_rel_z=-5,
        max_rel_z=5,
    )

    images = []

    for i, p in enumerate(positions):
        images.append(
            acquire_image(
                stage=stage,
                lights=lights,
                pos=np.append(p, z),
                read_qr=i == 0,
                crop_data=Rectangle(
                    top=CROP_TOP,
                    bottom=CAM_RES[1] - CROP_BOTTOM,
                    left=CROP_LEFT,
                    right=CAM_RES[0] - CROP_RIGHT,
                ),
            )
        )

    stage.move_position((TEMPLATE_LENGTH / 2, TEMPLATE_LENGTH / 2, 100))
    return images

In [ ]:
# list available serial ports
stage_port = None
ports = list_ports()
for port in ports:
    if "USB Serial" in port.description:
        stage_port = port
        print("* " + str(port))
    else:
        print("  " + str(port))

In [ ]:
stage = Stage(stage_port, 115200)

In [ ]:
lights = Enderlights()
lights.shutter(True)
time.sleep(1)
lights.shutter(False)

In [ ]:
stage.home()
stage.move_position((TEMPLATE_LENGTH / 2, TEMPLATE_LENGTH / 2, 100))
stage.finish_moves()

In [ ]:
preview().start_still()
preview().camera.set_controls(
    {"LensPosition": preview().camera.camera_controls["LensPosition"][1]}
)
preview().set_crop(top=CROP_TOP, bottom=CROP_BOTTOM, left=CROP_LEFT, right=CROP_RIGHT)

In [ ]:
preview().show()

In [ ]:
images = run_job(stage=stage, lights=lights, speed=6000, start_height=36)
len(images)

In [ ]:
sel_image = pn.widgets.IntSlider(
    name="Select image",
    start=0,
    end=len(images) - 1,
    value=0,
    sizing_mode="stretch_width",
)
ph_image = pn.pane.Placeholder()


@pn.depends(sel_image.param.value, watch=True)
def on_index_changed(index):
    ph_image.object = to_pil(images[index]).resize((600, 600))


on_index_changed(sel_image.value)

pn.Column(ph_image, sel_image)

In [ ]:
stage.set_speed_limit(100000, debug=True)

In [ ]:
stage.write_code("M503", debug=True)

In [ ]:
stage.set_speed(6000)
stage.move_relative(-100, -100)
stage.move_relative(100, 100)

In [ ]:
stage.move_axis("z", -50)